# Final pipeline (public-bounds version) + all paper figures and tables
What this notebook does, in one run:
1. **No data-dependent bounds.** A random 10% *public reference sample* is set aside; all clipping bounds and Naive-Bayes bin edges come from it. Models are trained and tested only on the remaining 90% (assumption: a small public sample exists; state it in the threat model).
2. **Matched total epsilon** (10, 25, 50, 100): central DP-LR, central DP-Gaussian-NB, local LR, local NB. Local per-attribute epsilon = total/(d+1) (features + label). No interpolation.
3. **Attacks:** linkage (re-identification) for several pool sizes, and membership inference (with an over-fitted RF positive control).
4. **Outputs** (Drive folder `dp_paper/final/`): CSVs, LaTeX tables, Wilcoxon tests, and publication figures (PNG 300 dpi + PDF). Everything is zipped to `paper_assets.zip` and downloaded.

Progress is saved per dataset, so after a disconnect just run all cells again. Optional extra datasets: fill `EXTRA_CSV` in cell 1.

In [ ]:
!pip -q install diffprivlib
import numpy as np, pandas as pd, warnings, time, os, glob, shutil
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import fetch_openml
from diffprivlib.models import LogisticRegression as DPLR, GaussianNB as DPGNB
warnings.filterwarnings("ignore")

try:
    from google.colab import drive, files
    drive.mount('/content/drive'); OUT = "/content/drive/MyDrive/dp_paper/final/"; IN_COLAB = True
except Exception:
    OUT = "./final/"; IN_COLAB = False
os.makedirs(OUT + "figures", exist_ok=True)

QUICK = os.environ.get("DP_QUICK") == "1"          # tiny smoke-test mode (ignore in Colab)
SEED = 42; MAX_ROWS = 5000; PUBLIC_FRAC = 0.10; NB_BINS = 5
TOTALS = [10, 25, 50, 100]                          # matched total epsilon
N_DRAWS = 5                                         # noise draws / model seeds
MIA_REPS = 10; REID_DRAWS = 3; POOLS = [100, 500, 2000]; REID_TOTALS = [10, 25, 50, 100]
if QUICK: TOTALS = [25, 50]; N_DRAWS = 2; MIA_REPS = 3; REID_DRAWS = 1; POOLS = [100, 500]; REID_TOTALS = [25, 50]

# optional extra datasets from CSV files you upload to Colab:  name: (path, target_column, positive_value)
# Lending Club (small, 9,578 loans, 2007-2010): download loan_data.csv from Kaggle, upload to Colab, then use:
#   EXTRA_CSV = {"lending_club": ("/content/loan_data.csv", "not.fully.paid", 1)}
EXTRA_CSV = {}

In [ ]:
# ---------------- dataset loaders ----------------
def load_loan():
    d = pd.read_csv("https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv")
    return d.drop(columns=["Loan_ID", "Loan_Status"]), (d.Loan_Status == "Y").astype(int).values
def load_german():
    g = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv", header=None)
    return g.iloc[:, :-1], (g.iloc[:, -1] == 2).astype(int).values
def load_adult():
    a = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame
    return a.drop(columns=["class"]), (a["class"] == ">50K").astype(int).values
def load_taiwan():
    t = fetch_openml(data_id=42477, as_frame=True, parser="auto").frame
    return t.iloc[:, :-1], pd.factorize(t.iloc[:, -1])[0].astype(int)
def load_openml(i):
    f = fetch_openml(data_id=i, as_frame=True, parser="auto").frame
    t = f.columns[-1]; lab = f[t].astype(str)
    return f.drop(columns=[t]), (lab != lab.mode()[0]).astype(int).values
def load_lending_club():
    d = pd.read_csv("https://raw.githubusercontent.com/Amitabh1989/random_forest/master/loan_data.csv")   # Lending Club 2007-2010, 9,578 loans
    return d.drop(columns=["not.fully.paid"]), d["not.fully.paid"].astype(int).values
def load_openml_xy(i):
    b = fetch_openml(data_id=i, as_frame=True, parser="auto")        # uses the dataset's own target column
    X = b.data; lab = b.target.astype(str)
    return X, (lab != lab.mode()[0]).astype(int).values              # positive class = minority (default)
def load_csv(path, target, positive=None):
    d = pd.read_csv(path); d = d.loc[:, ~d.columns.astype(str).str.contains("^Unnamed")]
    lab = d[target]
    y = (lab == positive).astype(int).values if positive is not None else (lab.astype(str) != lab.astype(str).mode()[0]).astype(int).values
    return d.drop(columns=[target]), y

ALL = {"loan": load_loan, "german": load_german, "adult": load_adult, "taiwan": load_taiwan,
       "credit_approval": lambda: load_openml(29), "bank_marketing": lambda: load_openml(1461), "diabetes": lambda: load_openml(37),
       "gmc": lambda: load_openml_xy(46468),
       "lending_club": load_lending_club}          # Give Me Some Credit (OpenML 46468, 45,000 rows, ~6.7% default)
MAX_ROWS_OVERRIDE = {"gmc": 15000}
for k, (p, t, pos) in EXTRA_CSV.items(): ALL[k] = (lambda p=p, t=t, pos=pos: load_csv(p, t, pos))
DATASETS = {k: ALL[k] for k in (["loan", "german", "lending_club"] if QUICK else list(ALL))}

def prepare(name):
    X, y = DATASETS[name](); y = np.asarray(y).astype(int)
    cap = MAX_ROWS_OVERRIDE.get(name, MAX_ROWS)
    if len(X) > cap:
        idx = np.random.default_rng(SEED).choice(len(X), cap, replace=False); X, y = X.iloc[idx].reset_index(drop=True), y[idx]
    X = X.copy(); num, cat = [], []
    for c in X.columns:
        if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 10: X[c] = X[c].astype(float); num.append(str(c))
        else: X[c] = pd.factorize(X[c].astype(str))[0]; cat.append(str(c))
    X.columns = X.columns.astype(str)
    perm = np.random.default_rng(SEED).permutation(len(X)); n_pub = max(int(PUBLIC_FRAC * len(X)), 50)
    Xpub = X.iloc[perm[:n_pub]].reset_index(drop=True); Xpriv = X.iloc[perm[n_pub:]].reset_index(drop=True); ypriv = y[perm[n_pub:]]
    for c in num:
        m = Xpub[c].median(); Xpub[c] = Xpub[c].fillna(m); Xpriv[c] = Xpriv[c].fillna(m)
    bounds = {}
    for c in num:
        lo, hi = float(Xpub[c].quantile(.01)), float(Xpub[c].quantile(.99))
        bounds[c] = (lo, hi if hi > lo else lo + 1.0)
    dom = {c: int(X[c].max()) + 1 for c in cat}
    edges = {c: np.quantile(Xpub[c], np.linspace(0, 1, NB_BINS + 1)[1:-1]) for c in num}
    ypub = y[perm[:n_pub]]
    return dict(name=name, X=Xpriv, y=ypriv, Xpub=Xpub, ypub=ypub, num=num, cat=cat, bounds=bounds, dom=dom, edges=edges, d=len(num) + len(cat))

In [ ]:
# ---------------- mechanisms and designs ----------------
def scaled(X, D, c):
    lo, hi = D["bounds"][c]; return (X[c].clip(lo, hi).values - lo) / (hi - lo)       # in [0,1]; sensitivity 1
def onehot(codes, K):
    M = np.zeros((len(codes), K)); M[np.arange(len(codes)), np.asarray(codes).astype(int)] = 1; return M
def rr_codes(col, k, eps, rng):
    col = np.asarray(col).astype(int)
    if k == 1: return col
    p = np.exp(eps) / (np.exp(eps) + k - 1); keep = rng.random(len(col)) < p
    r = rng.integers(0, k - 1, len(col)); return np.where(keep, col, r + (r >= col))
def chan(k, eps):
    p = np.exp(eps) / (np.exp(eps) + k - 1); q = (1 - p) / (k - 1); M = np.full((k, k), q); np.fill_diagonal(M, p); return M

def design(D, eps=None, rng=None):
    # eps=None -> clean design; otherwise Laplace(1/eps) on scaled numerics + randomized response on categoricals
    X = D["X"]; cols, names = [], []
    for c in D["num"]:
        v = scaled(X, D, c)
        if eps is not None: v = v + rng.laplace(0, 1 / eps, len(v))
        cols.append(v[:, None]); names.append(c)
    for c in D["cat"]:
        K = D["dom"][c]; codes = X[c].values.astype(int)
        if eps is not None: codes = rr_codes(codes, K, eps, rng)
        cols.append(onehot(codes, K)); names += [f"{c}_{k}" for k in range(K)]
    return pd.DataFrame(np.hstack(cols), columns=names)

def cv_auc(D, fit_score):
    y = D["y"]; a = []
    for tr, te in StratifiedKFold(10, shuffle=True, random_state=SEED).split(D["X"], y):
        a.append(roc_auc_score(y[te], fit_score(tr, te)))
    return float(np.mean(a))

def central(D, Z, make, reps):
    y = D["y"]; v = [cv_auc(D, lambda tr, te, s=s: make(s).fit(Z.iloc[tr], y[tr]).predict_proba(Z.iloc[te])[:, 1]) for s in range(reps)]
    return np.mean(v), np.std(v)

def local_lr(D, Zc, eps, rng):
    y = D["y"]; Zn = design(D, eps, rng); yn = rr_codes(y, 2, eps, rng)      # label randomized too
    def fs(tr, te):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)).fit(Zn.iloc[tr], yn[tr])
        return m.predict_proba(Zc.iloc[te])[:, 1]                           # clean test records
    return cv_auc(D, fs)

def local_nb(D, eps, rng):
    X = D["X"]; y = D["y"]
    Cc = {c: X[c].values.astype(int) for c in D["cat"]}
    for c in D["num"]: Cc[c] = np.clip(np.searchsorted(D["edges"][c], X[c].clip(*D["bounds"][c]).values), 0, NB_BINS - 1)
    K = {**D["dom"], **{c: NB_BINS for c in D["num"]}}
    Cn = {c: rr_codes(Cc[c], K[c], eps, rng) for c in Cc}; yn = rr_codes(y, 2, eps, rng)
    My = np.linalg.inv(chan(2, eps)); Mx = {c: np.linalg.inv(chan(K[c], eps)) for c in Cc if K[c] > 1}
    def fs(tr, te):
        ny = np.clip(My @ np.bincount(yn[tr], minlength=2), 1e-3, None); score = np.tile(np.log(ny / ny.sum()), (len(te), 1))
        for c in Cc:
            if K[c] == 1: continue
            N = np.zeros((2, K[c])); np.add.at(N, (yn[tr], Cn[c][tr]), 1)
            T = np.clip(My @ N @ Mx[c], 0, None) + 1.0; P = T / T.sum(1, keepdims=True)    # debiased joint counts + smoothing
            score += np.log(P[:, Cc[c][te]]).T
        return score[:, 1] - score[:, 0]
    return cv_auc(D, fs)

In [ ]:
# ---------------- experiment 1: matched-budget utility ----------------
def run_utility(D):
    y = D["y"]; d = D["d"]; Z = design(D); B = np.sqrt(d); rows = []
    meta = dict(dataset=D["name"], n_priv=len(y), d=d, n_num=len(D["num"]), n_cat=len(D["cat"]), majority=max(y.mean(), 1 - y.mean()))
    m, s = central(D, Z, lambda sd: LogisticRegression(C=1.0, max_iter=500), 1); rows.append(dict(**meta, method="LR_nonprivate", total_eps=np.nan, auc=m, sd=0.0))
    for t in TOTALS:
        m, s = central(D, Z, lambda sd: DPLR(epsilon=t, data_norm=B, C=1.0, random_state=sd), N_DRAWS); rows.append(dict(**meta, method="central_DPLR", total_eps=t, auc=m, sd=s))
        m, s = central(D, Z, lambda sd: DPGNB(epsilon=t, bounds=(0, 1), random_state=sd), N_DRAWS); rows.append(dict(**meta, method="central_DPGNB", total_eps=t, auc=m, sd=s))
        e = t / (d + 1)
        v = [local_lr(D, Z, e, np.random.default_rng(31 * k + t)) for k in range(N_DRAWS)]; rows.append(dict(**meta, method="local_LR", total_eps=t, auc=np.mean(v), sd=np.std(v)))
        v = [local_nb(D, e, np.random.default_rng(57 * k + t)) for k in range(N_DRAWS)]; rows.append(dict(**meta, method="local_NB", total_eps=t, auc=np.mean(v), sd=np.std(v)))
    return rows

# ---------------- experiment 2: linkage (re-identification) attack, several pool sizes ----------------
def run_reid(D):
    n = len(D["y"]); out = []
    for pool in [p for p in POOLS if p <= n]:
        idx = np.random.default_rng(SEED).choice(n, pool, replace=False); Xp = D["X"].iloc[idx].reset_index(drop=True); yp = D["y"][idx]
        xs = {c: scaled(Xp, D, c) for c in D["num"]}
        for tot in REID_TOTALS:
            e = tot / (D["d"] + 1); hits = []
            for dr in range(REID_DRAWS):
                rng = np.random.default_rng(77 * dr + tot); LL = np.zeros((pool, pool))
                for c in D["num"]:
                    z = xs[c] + rng.laplace(0, 1 / e, pool); LL += -e * np.abs(z[:, None] - xs[c][None, :]) - np.log(2 / e)
                for c in D["cat"]:
                    k = D["dom"][c]
                    if k == 1: continue
                    col = Xp[c].values.astype(int); z = rr_codes(col, k, e, rng); p = np.exp(e) / (np.exp(e) + k - 1); q = (1 - p) / (k - 1)
                    LL += np.where(z[:, None] == col[None, :], np.log(p), np.log(q))
                zy = rr_codes(yp, 2, e, rng); p2 = np.exp(e) / (np.exp(e) + 1)
                LL += np.where(zy[:, None] == yp[None, :], np.log(p2), np.log(1 - p2))
                diag = np.diag(LL); mx = LL.max(1); ties = (LL >= mx[:, None] - 1e-9).sum(1)
                hits.append(np.mean((diag >= mx - 1e-9) / ties))
            out.append(dict(dataset=D["name"], pool=pool, total_eps=tot, eps_attr=e, top1=np.mean(hits), sd=np.std(hits), random=1 / pool))
    return out

# ---------------- experiment 3: membership inference against released central models ----------------
def run_mia(D):
    y = D["y"]; n = len(y); Z = design(D); B = np.sqrt(D["d"]); mt = min(n // 2, 1000)
    makers = {"LR_nonprivate": lambda s: LogisticRegression(C=1.0, max_iter=500)}
    for t in TOTALS: makers[f"DPLR_eps{t}"] = (lambda t: (lambda s: DPLR(epsilon=t, data_norm=B, C=1.0, random_state=s)))(t)
    makers["RF_overfit_control"] = lambda s: RandomForestClassifier(300, random_state=s, n_jobs=-1)
    res = {k: [] for k in makers}
    for r in range(MIA_REPS):
        perm = np.random.default_rng(1000 + r).permutation(n); mem, non = perm[:mt], perm[mt:2 * mt]; lab = np.r_[np.ones(len(mem)), np.zeros(len(non))]
        for k, mk in makers.items():
            m = mk(r).fit(Z.iloc[mem], y[mem])
            conf = np.r_[m.predict_proba(Z.iloc[mem])[np.arange(len(mem)), y[mem]], m.predict_proba(Z.iloc[non])[np.arange(len(non)), y[non]]]
            res[k].append(roc_auc_score(lab, conf))
    return [dict(dataset=D["name"], model=k, mia_auc=np.mean(v), sd=np.std(v, ddof=1), ci95=1.96 * np.std(v, ddof=1) / np.sqrt(len(v))) for k, v in res.items()]

for name in DATASETS:
    todo = [(f"util_{name}.csv", run_utility), (f"reid_{name}.csv", run_reid), (f"mia_{name}.csv", run_mia)]
    if all(os.path.exists(OUT + f) for f, _ in todo): print(name, "already done"); continue
    try:
        t0 = time.time(); D = prepare(name)
        for f, fn in todo:
            if not os.path.exists(OUT + f): pd.DataFrame(fn(D)).to_csv(OUT + f, index=False)
        print(name, "done in", round(time.time() - t0), "s  (n_private =", len(D["y"]), ", d =", D["d"], ")")
    except Exception as ex:
        import traceback; traceback.print_exc(); print(name, "FAILED:", repr(ex))

In [ ]:
# ---------------- tables, statistics, LaTeX ----------------
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
U = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "util_*.csv"))], ignore_index=True)
RE = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "reid_*.csv"))], ignore_index=True)
MI = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "mia_*.csv"))], ignore_index=True)
ORDER = ["LR_nonprivate", "local_NB", "local_LR", "central_DPLR", "central_DPGNB"]
npv = U[U.method == "LR_nonprivate"].set_index("dataset").auc
tabs = {}
for t in TOTALS:
    tb = U[U.total_eps == t].pivot_table(index="dataset", columns="method", values="auc"); tb.insert(0, "LR_nonprivate", npv)
    tabs[t] = tb[[c for c in ORDER if c in tb.columns]]; print(f"\n=== AUC at total epsilon = {t} ==="); print(tabs[t].round(3).to_string())
    tabs[t].round(3).to_csv(OUT + f"table_auc_eps{t}.csv")
    try: open(OUT + f"table_auc_eps{t}.tex", "w").write(tabs[t].round(3).to_latex())
    except Exception as ex: print("latex export skipped:", ex)
print("\n=== Wilcoxon (central DP-LR vs local estimators) ===")
for t in [x for x in [25, 50] if x in tabs]:
    for other in ["local_NB", "local_LR"]:
        d_ = tabs[t][["central_DPLR", other]].dropna()
        if len(d_) >= 5: print(f"eps={t}: central_DPLR vs {other}: central higher in {(d_.iloc[:,0] > d_.iloc[:,1]).sum()}/{len(d_)}, p={stats.wilcoxon(d_.iloc[:,0], d_.iloc[:,1]).pvalue:.4f}")
print("\n=== Linkage attack: top-1 rate (pool x total eps) ===")
print(RE.pivot_table(index=["dataset", "pool"], columns="total_eps", values="top1").round(3).to_string())
print("\n=== Membership inference AUC ===")
print(MI.pivot_table(index="dataset", columns="model", values="mia_auc").round(3).to_string())
RE.to_csv(OUT + "table_reid_all.csv", index=False); MI.to_csv(OUT + "table_mia_all.csv", index=False)

In [ ]:
# ---------------- figures (PNG 300 dpi + PDF) ----------------
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5, "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300})
COL = {"central_DPLR": "#0072B2", "central_DPGNB": "#56B4E9", "local_LR": "#009E73", "local_NB": "#D55E00"}
LAB = {"central_DPLR": "Central DP-LR", "central_DPGNB": "Central DP-GNB", "local_LR": "Local LR", "local_NB": "Local NB"}
def save(fig, name): fig.savefig(OUT + f"figures/{name}.png", bbox_inches="tight"); fig.savefig(OUT + f"figures/{name}.pdf", bbox_inches="tight"); plt.show()
dsets = list(U.dataset.unique()); pal = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000", "#999999", "#8c564b", "#17becf"]

# A. AUC vs total epsilon, one panel per dataset
nc = 4; nr = int(np.ceil((len(dsets) + 1) / nc)); fig, axs = plt.subplots(nr, nc, figsize=(7.2, 1.9 * nr), sharex=True); axs = axs.ravel()
for ax, ds in zip(axs, dsets):
    g = U[U.dataset == ds]
    for m in ["central_DPLR", "central_DPGNB", "local_LR", "local_NB"]:
        gg = g[g.method == m].sort_values("total_eps"); ax.errorbar(gg.total_eps, gg.auc, yerr=gg.sd, marker="o", ms=3, lw=1.1, capsize=1.5, color=COL[m])
    ax.axhline(npv[ds], ls="--", c="k", lw=.8); ax.set_xscale("log"); ax.set_title(ds); ax.set_xticks(TOTALS); ax.set_xticklabels(TOTALS)
for ax in axs[len(dsets):]: ax.axis("off")
h = [plt.Line2D([], [], color=COL[m], marker="o", ms=3, label=LAB[m]) for m in COL] + [plt.Line2D([], [], color="k", ls="--", label="No privacy (LR)")]
axs[len(dsets)].legend(handles=h, loc="center", frameon=False)
fig.supxlabel("Total privacy budget ε_tot", fontsize=8); fig.supylabel("AUC (10-fold CV)", fontsize=8); fig.tight_layout(); save(fig, "fig_auc_curves")

# B. matched-budget bars at eps = 25 and 50
sel = [t for t in [25, 50] if t in tabs]; fig, axs = plt.subplots(1, len(sel), figsize=(3.6 * len(sel), 3.2), sharey=True); axs = np.atleast_1d(axs)
x = np.arange(len(dsets)); w = .2
for ax, t in zip(axs, sel):
    for k, m in enumerate(["local_NB", "local_LR", "central_DPLR", "central_DPGNB"]):
        ax.bar(x + (k - 1.5) * w, [tabs[t].loc[d_, m] for d_ in dsets], w, color=COL[m], label=LAB[m])
    ax.plot(x, [npv[d_] for d_ in dsets], "k_", ms=11, mew=1.2, label="No privacy (LR)")
    ax.set_xticks(x); ax.set_xticklabels(dsets, rotation=40, ha="right"); ax.set_title(f"Total ε = {t}", pad=4); ax.set_ylim(0.5, 1.0)
axs[0].set_ylabel("AUC (axis starts at 0.5)"); hh, ll = axs[0].get_legend_handles_labels()
fig.legend(hh, ll, loc="upper center", ncol=5, frameon=False, bbox_to_anchor=(0.5, 1.05)); fig.tight_layout(rect=(0, 0, 1, .93)); save(fig, "fig_matched_budget")

# C. linkage attack: (a) top-1 vs total eps (pool 500), (b) effect of pool size at the middle budget
p0 = 500 if 500 in RE.pool.unique() else int(RE.pool.max()); fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0))
for ds, c in zip(dsets, pal):
    g = RE[(RE.dataset == ds) & (RE.pool == p0)].sort_values("total_eps"); axs[0].plot(g.total_eps, g.top1, marker="o", ms=3, lw=1.1, color=c, label=ds)
axs[0].axhline(1 / p0, ls=":", c="grey"); axs[0].set_xscale("log"); axs[0].set_yscale("log"); axs[0].set_xlabel("Total privacy budget ε_tot"); axs[0].set_ylabel(f"Re-identification top-1 rate (pool = {p0})"); axs[0].legend(frameon=False, fontsize=6)
tm = 50 if 50 in REID_TOTALS else REID_TOTALS[-1]
for ds, c in zip(dsets, pal):
    g = RE[(RE.dataset == ds) & (RE.total_eps == tm)].sort_values("pool"); axs[1].plot(g.pool, g.top1, marker="o", ms=3, lw=1.1, color=c)
pools = sorted(RE.pool.unique()); axs[1].plot(pools, [1 / p for p in pools], ls=":", c="grey"); axs[1].set_xscale("log"); axs[1].set_yscale("log")
axs[1].set_xlabel("Candidate pool size"); axs[1].set_ylabel(f"Top-1 rate at ε_tot = {tm}"); fig.tight_layout(); save(fig, "fig_reid")

# D. membership inference
M = MI.pivot_table(index="dataset", columns="model", values="mia_auc"); E = MI.pivot_table(index="dataset", columns="model", values="ci95")
cols = [c for c in M.columns if c == "LR_nonprivate" or c.startswith("DPLR")] + ["RF_overfit_control"]; fig, ax = plt.subplots(figsize=(7.2, 2.9)); w = .8 / len(cols)
pp = ["#7f7f7f", "#0072B2", "#56B4E9", "#009E73", "#E69F00", "#CC79A7"]
for k, c in enumerate(cols): ax.bar(np.arange(len(M)) + (k - (len(cols) - 1) / 2) * w, M[c].values, w, yerr=E[c].values, capsize=1, color=pp[k % len(pp)], label=c.replace("_", " "))
ax.axhline(.5, ls=":", c="k", lw=.8); ax.set_ylim(.45, max(.85, M.values.max() + .03)); ax.set_xticks(np.arange(len(M))); ax.set_xticklabels(M.index); ax.set_ylabel("Membership-inference AUC (axis starts at 0.45)")
ax.legend(frameon=False, ncol=3, fontsize=6.5, loc="upper left"); fig.tight_layout(); save(fig, "fig_mia")

# E. truthfulness of randomized response (analytic) with the per-attribute epsilon implied by eps_tot = 50
e = np.linspace(0, 8, 400); fig, ax = plt.subplots(figsize=(3.6, 3.0))
for k, ls in [(2, "-"), (4, "--"), (8, ":")]: ax.plot(e, np.exp(e) / (np.exp(e) + k - 1), ls, c="k", lw=1.2, label=f"k = {k} categories")
dd = U.drop_duplicates("dataset").set_index("dataset").d; ea = [50 / (v + 1) for v in dd.values]
ax.plot(ea, [np.exp(x) / (np.exp(x) + 1) for x in ea], "o", c="#D55E00", ms=4, label="ε_tot = 50, binary attribute")
ax.set_xlabel("Per-attribute ε"); ax.set_ylabel("P(true category reported)"); ax.set_ylim(0, 1.02); ax.legend(frameon=False, loc="lower right"); fig.tight_layout(); save(fig, "fig_truthfulness")

# ---------------- zip everything ----------------
shutil.make_archive(OUT.rstrip("/") + "_paper_assets", "zip", OUT); print("zip:", OUT.rstrip("/") + "_paper_assets.zip")
if IN_COLAB: files.download(OUT.rstrip("/") + "_paper_assets.zip")